# GWL Training Notebook - Paperspace

Training notebook for Global Workspace LM on Paperspace Gradient/Core.

## Setup

In [ ]:
# Install dependencies
!pip install torch torchvision torchaudio
!pip install transformers datasets tiktoken
!pip install tqdm wandb

## Imports and Configuration

In [ ]:
import os
import sys
import json
import time
from pathlib import Path
from typing import Optional, Dict, Any, List

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## Load GWL Modules

In [ ]:
from gwl import GWLConfig, GlobalWorkspaceLM
from gwl.trainer import GWLTrainer, TextDataset

## Data Loading

Load training data from HuggingFace datasets.

In [ ]:
from datasets import load_dataset
import tiktoken

# Configuration
DATASET_NAME = "EleutherAI/pile"  # Or "wikitext", "c4", etc.
TOKENIZER_NAME = "gpt2"  # Use GPT-2 tokenizer
SEQ_LEN = 128
MAX_TRAIN_TOKENS = 100_000_000  # 100M tokens for training

# Load tokenizer
print(f"Loading tokenizer: {TOKENIZER_NAME}")
tokenizer = tiktoken.get_encoding(TOKENIZER_NAME)
VOCAB_SIZE = tokenizer.n_vocab
print(f"Vocabulary size: {VOCAB_SIZE}")

In [ ]:
# Load dataset
print(f"Loading dataset: {DATASET_NAME}")

# Option 1: Load from EleutherAI/pile (full dataset)
try:
    dataset = load_dataset(DATASET_NAME, split="train[:10%]")  # Start with 10%
    print(f"Loaded {len(dataset)} examples from pile")
except:
    # Option 2: Load WikiText
    print("Falling back to wikitext-103")
    dataset = load_dataset("wikitext", "wikitext-103-raw-v1", split="train")
    print(f"Loaded {len(dataset)} examples from wikitext")

print(f"Dataset sample: {dataset[0]['text'][:100]}...")

In [ ]:
def tokenize_dataset(dataset, tokenizer, max_tokens=MAX_TRAIN_TOKENS):
    """Tokenize dataset and create training sequences."""
    print("Tokenizing dataset...")
    
    all_tokens = []
    total_tokens = 0
    
    for i, example in enumerate(tqdm(dataset, desc="Tokenizing")):
        if total_tokens >= max_tokens:
            break
        
        text = example.get('text', example.get('content', ''))
        if not text:
            continue
        
        tokens = tokenizer.encode(text)
        all_tokens.extend(tokens)
        total_tokens += len(tokens)
    
    print(f"Total tokens: {len(all_tokens):,}")
    return all_tokens

# Tokenize
tokens = tokenize_dataset(dataset, tokenizer)

In [ ]:
# Create datasets
print("Creating datasets...")

train_tokens = tokens[:-10000]  # Reserve 10k for validation
val_tokens = tokens[-10000:]

train_dataset = TextDataset(train_tokens, seq_len=SEQ_LEN, stride=SEQ_LEN//2)
val_dataset = TextDataset(val_tokens, seq_len=SEQ_LEN, stride=SEQ_LEN)

print(f"Training sequences: {len(train_dataset):,}")
print(f"Validation sequences: {len(val_dataset):,}")

## Model Configuration

In [ ]:
# Model configuration - tuned for Paperspace GPU
# Using a smaller config to start, can scale up

VOCAB_SIZE = 50257  # GPT-2 vocab size
EMBED_DIM = 256
WORKSPACE_DIM = 512
NUM_PROCESSORS = 32  # Start with 32, can increase
PROCESSOR_DIM = 128
NUM_STEPS = 3

# Create config
gwl_config = GWLConfig(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    workspace_dim=WORKSPACE_DIM,
    num_processors=NUM_PROCESSORS,
    processor_config__hidden_dim=PROCESSOR_DIM,
    num_steps=NUM_STEPS,
    use_residual_workspace=True,
    use_layer_norm=True,
    use_dropout=True,
)

print(f"Config created with vocab_size={VOCAB_SIZE}")

## Create Model

In [ ]:
# Create model
print("Creating model...")
model = GlobalWorkspaceLM(gwl_config)
model = model.to(device)

# Count parameters
num_params = sum(p.numel() for p in model.parameters())
num_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {num_params:,}")
print(f"Trainable parameters: {num_trainable:,}")

# Estimate model size
model_size_mb = num_params * 4 / (1024 ** 2)  # Assuming float32
print(f"Model size: {model_size_mb:.2f} MB")

## Training Configuration

In [ ]:
# Training hyperparameters
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
WARMUP_STEPS = 500
MAX_STEPS = 10000  # Adjust based on time budget
EVAL_INTERVAL = 500
LOG_INTERVAL = 50
SAVE_INTERVAL = 1000

# Create training config
from gwl.config import TrainingConfig

train_config = TrainingConfig(
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    grad_clip=GRAD_CLIP,
    warmup_steps=WARMUP_STEPS,
    max_steps=MAX_STEPS,
    eval_interval=EVAL_INTERVAL,
    log_interval=LOG_INTERVAL,
    save_interval=SAVE_INTERVAL,
    checkpoint_dir="checkpoints",
    use_amp=True,
)

print("Training config:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Max steps: {MAX_STEPS}")

## Create DataLoaders

In [ ]:
# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

## Training Loop

In [ ]:
from torch.optim.lr_scheduler import LambdaLR

# Create optimizer
optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

# Learning rate scheduler
def lr_lambda(step):
    if step < WARMUP_STEPS:
        return step / max(1, WARMUP_STEPS)
    progress = (step - WARMUP_STEPS) / max(1, MAX_STEPS - WARMUP_STEPS)
    return max(0.1, 0.5 * (1.0 + torch.cos(torch.pi * progress)))

scheduler = LambdaLR(optimizer, lr_lambda)

# Mixed precision scaler
scaler = GradScaler() if train_config.use_amp else None

# Training loop
print("Starting training...")
global_step = 0
best_val_loss = float('inf')

train_iter = iter(train_loader)
train_losses = []

while global_step < MAX_STEPS:
    # Get batch
    try:
        batch = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        batch = next(train_iter)
    
    input_ids, target_ids = batch
    input_ids = input_ids.to(device, non_blocking=True)
    target_ids = target_ids.to(device, non_blocking=True)
    
    # Forward pass
    model.train()
    
    if scaler:
        with autocast():
            outputs = model(input_ids, target_ids)
            loss = outputs["loss"]
    else:
        outputs = model(input_ids, target_ids)
        loss = outputs["loss"]
    
    if loss is None:
        continue
    
    # Backward pass
    if scaler:
        scaler.scale(loss).backward()
        if GRAD_CLIP > 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
    else:
        loss.backward()
        if GRAD_CLIP > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
    
    optimizer.zero_grad()
    scheduler.step()
    
    train_losses.append(loss.item())
    global_step += 1
    
    # Logging
    if global_step % LOG_INTERVAL == 0:
        avg_loss = sum(train_losses[-LOG_INTERVAL:]) / len(train_losses[-LOG_INTERVAL:])
        lr = scheduler.get_last_lr()[0]
        print(f"Step {global_step}/{MAX_STEPS} | Loss: {avg_loss:.4f} | LR: {lr:.6f}")
    
    # Evaluation
    if global_step % EVAL_INTERVAL == 0:
        model.eval()
        val_losses = []
        
        with torch.no_grad():
            for val_batch in val_loader:
                input_ids, target_ids = val_batch
                input_ids = input_ids.to(device)
                target_ids = target_ids.to(device)
                
                outputs = model(input_ids, target_ids)
                if outputs["loss"] is not None:
                    val_losses.append(outputs["loss"].item())
        
        val_loss = sum(val_losses) / len(val_losses) if val_losses else 0
        print(f"Step {global_step} | Val Loss: {val_loss:.4f}")
        
        # Save best
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save({
                "step": global_step,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_loss,
            }, "checkpoints/best.pt")
            print(f"Saved best model (val_loss: {val_loss:.4f})")
        
        model.train()
    
    # Checkpoint
    if global_step % SAVE_INTERVAL == 0:
        torch.save({
            "step": global_step,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
        }, f"checkpoints/step_{global_step}.pt")
        print(f"Saved checkpoint at step {global_step}")

print("Training complete!")
print(f"Best validation loss: {best_val_loss:.4f}")

## Evaluation

In [ ]:
# Load best model
checkpoint = torch.load("checkpoints/best.pt")
model.load_state_dict(checkpoint["model_state_dict"])
print(f"Loaded best model from step {checkpoint['step']}")

# Generate some text
model.eval()

# Test prompt
prompt = "The quick brown fox"
prompt_ids = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long).to(device)

with torch.no_grad():
    generated = model.generate(
        prompt_ids,
        max_new_tokens=50,
        temperature=0.8,
        top_k=50,
    )

output_text = tokenizer.decode(generated[0].cpu().tolist())
print(f"\nPrompt: {prompt}")
print(f"Generated: {output_text}")

## Next Steps

1. **Scale up model**: Increase num_processors, processor_dim, workspace_dim
2. **More training**: Increase MAX_STEPS for better convergence
3. **Better data**: Use full Pile dataset or larger text corpora
4. **Evaluation**: Run on GSM8K, BIG-Bench benchmarks
5. **Hyperparameter search**: Try different processor types, competition mechanisms